<p><font size="6" color="grey"><b>Machine Learning</b></font></p>


<p><font size="5" color="grey"><b>XAI Advanced – Fairness, Robustheit, Interaktionen & mehr</b></font></p>

---

**Voraussetzung:** Grundlagen der XAI-Methoden

Dieses Notebook vertieft sieben fortgeschrittene XAI-Themen:

| | Thema | Warum relevant? | Komplexität |
|-|-------|-----------------|-------------|
| **A** | Fairness & Bias | sex/pclass dominieren – ethische Bewertung des Modells | ⭐⭐⭐ |
| **B** | Robustheit der Erklärungen | LIME/SHAP können je nach Seed schwanken | ⭐⭐⭐ |
| **C** | Surrogate Models & EBM | Direktinterpretierbare Alternativen zur Black Box | ⭐⭐⭐ |
| **D** | SHAP Interaction Values | Wechselwirkungen zwischen Features sichtbar machen | ⭐⭐⭐⭐ |
| **E** | Kausalität vs. Korrelation | XAI erklärt Modelllogik, keine Kausalzusammenhänge | ⭐⭐ |
| **F** | Anchors | Wenn-dann-Regeln für stabile lokale Erklärungen | ⭐⭐⭐ |
| **G** | Example-Based Explanations | Erklärung über ähnliche Fälle und Prototypen | ⭐⭐⭐ |


# 0 | Install & Import
---

In [ ]:
# Install
!uv pip install -q shap lime eli5 interpret pyale
!uv pip install -q git+https://github.com/parrt/dtreeviz.git
!uv pip install -q SALib
!uv pip install -q shap lime interpret fairlearn alibi

In [ ]:
# Daten & Strukturen
from pandas import read_excel, DataFrame  # Excel laden und Daten verwalten
import numpy as np  # Numerische Operationen

# Datenvorverarbeitung
from sklearn.preprocessing import OrdinalEncoder  # Kategoriale Variablen encodieren

# Datenaufteilung
from sklearn.model_selection import train_test_split  # Train/Test-Split

# Modell
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import NearestNeighbors
from sklearn.tree import DecisionTreeClassifier

# Evaluation & Metriken
from sklearn.metrics import accuracy_score  # Klassifikationsmetriken

# XAI Frameworks
import shap  # SHAP-Erklärungen
from lime.lime_tabular import LimeTabularExplainer  # Tabellarische LIME-Erklärungen
import eli5  # Modellinterpretation
from eli5.sklearn import PermutationImportance  # Feature-Importance via Permutation
from interpret import show  # Interpretierbarkeits-Visualisierung
from interpret.blackbox import ShapKernel  # SHAP Kernel Explainer
from interpret.glassbox import ExplainableBoostingClassifier
from fairlearn.metrics import (
    MetricFrame, selection_rate,
    demographic_parity_difference, equalized_odds_difference
)

# Statistisches Framework
from SALib.sample import saltelli
from SALib.analyze import sobol

# Visualisierung
import plotly.express as px  # Interaktive Plots
import plotly.graph_objects as go
import matplotlib.pyplot as plt  # Klassische Visualisierung
import dtreeviz  # Baumvisualisierung

In [ ]:
POS = '#636EFA'
NEG = '#EF553B'

# 1  | Understand
---

<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Aufgabe verstehen</br>
✅ Daten sammeln</br>
✅ Statistische Analyse (Min, Max, Mean, Korrelation, ...)</br>
✅ Datenvisualisierung (Streudiagramm, Box-Plot, ...)</br>
✅ Prepare Schritte festlegen</br>

<p><font color='black' size="5">
📒 Anwendungsfall
</font></p>

In [ ]:
df = read_excel(
    "https://raw.githubusercontent.com/ralf-42/ML_Intro/main/02_daten/05_tabellen/titanic.xlsx",
    usecols=["pclass", "survived", "sex", "age", "sibsp", "parch"],
)

data = df.copy()
target = data.pop("survived")

data.head()

#  2 | Prepare
---


<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Train-Test-Split durchführen</br>
✅ Nicht benötigte Features löschen</br>
✅ Datentyp ermitteln/ändern</br>
✅ Missing Values behandeln</br>
✅ Ausreißer behandeln</br>
✅ Kategorischer Features Kodieren</br>
✅ Numerischer Features skalieren</br>
✅ Feature-Engineering (neue Features schaffen)</br>
✅ Dimensionalität reduzieren</br>
✅ Resampling (Over-/Undersampling)</br>
✅ Pipeline erstellen/konfigurieren</br>


<font color='black' size="5">
Datentyp ermitteln
</font>


In [ ]:
all_col = data.columns
num_col = data.select_dtypes(include="number").columns
cat_col = data.select_dtypes(exclude="number").columns

<font color='black' size="5">
Train-Test-Split
</font>


In [ ]:
data_train, data_test, target_train, target_test = train_test_split(
    data, target, test_size=0.20, stratify=target, random_state=42)

<font color='black' size="5">
Kodierung
</font>

In [ ]:
encoder = OrdinalEncoder()
encoder.fit(data_train[cat_col])
data_train[cat_col] = encoder.transform(data_train[cat_col])
data_test[cat_col] = encoder.transform(data_test[cat_col])

# 3 | Modeling
---

<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Modellauswahl treffen</br>
✅ Pipeline erweitern/konfigurieren</br>
✅ Training durchführen</br>
✅ Hyperparameter Tuning</br>
✅ Cross-Validiation</br>
✅ Bootstrapping</br>
✅ Regularization</br>

<p><font color='black' size="5">
🏃 Modellauswahl & Training
</font></p>


In [ ]:
model = RandomForestClassifier(
    n_estimators=200,        # Mehr Bäume für stabilere Vorhersagen
    max_depth=5,             # Weniger tief = weniger Overfitting
    min_samples_split=10,    # Mind. 10 Samples für Split
    min_samples_leaf=5,      # Mind. 5 Samples pro Blatt
    max_features='sqrt',     # Nur Wurzel der Features pro Split
    random_state=42
)

model.fit(data_train, target_train)

# 4 | Evaluate
---

<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Prognose (Train, Test) erstellen</br>
✅ Modellgüte prüfen</br>
✅ Residuenanalyse erstellen</br>
✅ Feature Importance/Selektion prüfen</br>
✅ Robustheitstest erstellen</br>
✅ Modellinterpretation erstellen</br>
✅ Sensitivitätsanalyse erstellen</br>
✅ Kommunikation (Key Takeaways)</br>

<p><font color='black' size="5">
🎯 Accuracy
</font></p>


In [ ]:
target_train_pred = model.predict(data_train)
target_test_pred = model.predict(data_test)

acc_train = accuracy_score(target_train, target_train_pred) * 100
acc_test = accuracy_score(target_test, target_test_pred) * 100

print(f"Train Accuracy: {acc_train:5.2f}%")
print(f"Test Accuracy:  {acc_test:5.2f}%")

<p><font color='black' size="5">
📝 Testpersonen: Rose & Jack
</font></p>

In [ ]:
# Rose DeWitt Bukater (Kate Winslet) - Original-Werte (keine Skalierung!)
rose = DataFrame(
    {"age": [22], "sex": [0], "sibsp": [0], "parch": [1], "pclass": [1]},  # sex=0 (female)
    index=["Rose"],
)

# Jack Dawson (Leonardo DiCaprio) - Original-Werte (keine Skalierung!)
jack = DataFrame(
    {"age": [23], "sex": [1], "sibsp": [0], "parch": [0], "pclass": [3]},  # sex=1 (male)
    index=["Jack"],
)

# Spaltenreihenfolge an Trainingsdaten angleichen
rose = rose[data_train.columns]
jack = jack[data_train.columns]

rose_pred = model.predict_proba(rose)[0][1] * 100
jack_pred = model.predict_proba(jack)[0][1] * 100

print(f"Rose Ueberlebenschance: {rose_pred:.2f}%")
print(f"   (22 Jahre, weiblich, 1. Klasse)")
print(f"Jack Ueberlebenschance: {jack_pred:.2f}%")
print(f"   (23 Jahre, maennlich, 3. Klasse)")

Das Modell gibt für Rose eine Überlebenschance von **~62%** aus —
deutlich über dem Durchschnitt (26%). Für Jack sind es nur **~7%** —
weit unter dem Durchschnitt.

Diese Ausgangspunkte werden in den folgenden XAI-Abschnitten erklärt:
Was sind die Gründe für diesen ~55 Prozentpunkte-Unterschied?


<p><font color='black' size="5">
🌳 Entscheidungsbaum Visualisierung
</font></p>

**Hinweis**: RandomForest besteht aus 200 Bäumen. Hier wird nur der ersten Baum als Beispiel visualisiert.


In [ ]:
# Extrahiere den ersten Baum aus dem RandomForest
single_tree = model.estimators_[1]

# Erstelle dtreeviz Visualisierung für diesen einzelnen Baum
viz_model = dtreeviz.model(
    single_tree,
    X_train=data_train,
    y_train=target_train,
    target_name="survived",
    class_names=["not survived", "survived"],
    feature_names=list(data_train.columns),
)

print(f"✅ Visualisierung für Baum 1 von {model.n_estimators} erstellt")

In [ ]:
# Local Explanation für Rose - total
one = rose.iloc[0].values
viz_model.view(x=one, scale=1.0, fontname="Monospace")

Der vollständige Baum zeigt alle Entscheidungspfade: Farbige Knoten = Mehrheit der Klasse,
Breite der Knoten = Anzahl Samples, Pfad für Rose (22, weiblich, 1. Klasse) ist hervorgehoben.

> Hinweis: Dies ist nur **Baum 1** von 200 — jeder Baum im RandomForest trifft leicht
> unterschiedliche Entscheidungen. Die finale Vorhersage ist der Durchschnitt aller 200 Bäume.


In [ ]:
# local Explanation - single
viz_model.view(x=one, scale=1.2, show_just_path=True, fontname="Monospace")

Der Entscheidungspfad zeigt nur die Knoten, die für Roses Vorhersage relevant sind:
Das Modell prüft zuerst das Geschlecht, dann die Klasse — und kommt damit direkt
zur hohen Überlebenswahrscheinlichkeit. Die restlichen Features spielen auf diesem Pfad keine Rolle.


# 5 | Deploy
---

<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Modellexport und -speicherung</br>
✅ Abhängigkeiten und Umgebung</br>
✅ Sicherheit und Datenschutz</br>
✅ In die Produktion integrieren</br>
✅ Tests und Validierung</br>
✅ Dokumentation & Wartung</br>

# 🧑‍🎓 XAI Frameworks
---


# A | Fairness & Bias ⚖️
---

**Was ist Fairness-Analyse?**

Fairness-Metriken prüfen, ob ein Modell bestimmte Gruppen systematisch benachteiligt.
Im Titanic-Datensatz dominieren `sex` und `pclass` — historisch korrekt, aber als
Produktionsmodell ethisch problematisch.

| Metrik | Bedeutung |
|--------|-----------|
| **Selection Rate** | Anteil positiver Vorhersagen pro Gruppe |
| **Demographic Parity Difference** | Unterschied der Selection Rate zwischen Gruppen |
| **Equalized Odds Difference** | Unterschied in TPR und FPR zwischen Gruppen |

Grenze: Fairness-Metriken widersprechen sich oft — es ist mathematisch unmöglich,
alle gleichzeitig zu erfüllen (Fairness-Impossibility-Theorem).


In [ ]:
# Fairness nach Geschlecht
sens_sex = data_test['sex'].map({0.0: 'weiblich', 1.0: 'maennlich'})

mf_sex = MetricFrame(
    metrics={'Accuracy': accuracy_score, 'Selection Rate': selection_rate},
    y_true=target_test,
    y_pred=target_test_pred,
    sensitive_features=sens_sex
)

print('Fairness-Metriken nach Geschlecht:')
display(mf_sex.by_group)
print()
dpd = demographic_parity_difference(target_test, target_test_pred, sensitive_features=sens_sex)
eod = equalized_odds_difference(target_test, target_test_pred, sensitive_features=sens_sex)
print(f'Demographic Parity Difference: {dpd:.4f}')
print(f'Equalized Odds Difference:     {eod:.4f}')


| Gruppe | Accuracy | Selection Rate |
|--------|----------|----------------|
| männlich | **86.3%** | **0.6%** |
| weiblich | **64.7%** | **53.9%** |

- Das Modell sagt für Männer fast nie 'Survived' (0.6% Selection Rate) — für Frauen in
  über der Hälfte der Fälle (53.9%). Dieser Unterschied von 53.3 Prozentpunkten ist der **DPD**.
- Der **EOD = 0.70** zeigt: True-Positive-Rate und False-Positive-Rate weichen zwischen
  Männern und Frauen stark ab — das Modell macht für beide Gruppen sehr unterschiedlich viele Fehler.
- Hohe Männer-Accuracy (86.3%) ist irreführend: Das Modell sagt fast immer 'nicht überlebt'
  — was für Männer statistisch oft stimmt, aber kein echtes Verständnis zeigt.


In [ ]:
# Selection Rate nach Passagierklasse
# pclass ist numerisch (kein OrdinalEncoding) → Werte 1, 2, 3
sens_pclass = data_test['pclass'].round().astype(int).map(
    {1: '1. Klasse', 2: '2. Klasse', 3: '3. Klasse'}
).fillna('unbekannt')

mf_pclass = MetricFrame(
    metrics={'Accuracy': accuracy_score, 'Selection Rate': selection_rate},
    y_true=target_test,
    y_pred=target_test_pred,
    sensitive_features=sens_pclass
)

df_fair = mf_pclass.by_group.reset_index()
df_fair.columns = ['Klasse', 'Accuracy', 'Selection Rate']

# Farbe pro Klasse (diskret) statt kontinuierlicher Farbskala
farben = {'1. Klasse': '#636EFA', '2. Klasse': '#EF553B', '3. Klasse': '#00CC96'}

fig = px.bar(
    df_fair, x='Klasse', y='Selection Rate',
    title='Selection Rate (Vorhersage: Survived) nach Passagierklasse',
    color='Klasse',
    color_discrete_map=farben,
    labels={'Selection Rate': 'Selection Rate', 'Klasse': 'Passagierklasse'},
    text_auto='.1%'
)
fig.add_hline(y=mf_pclass.overall['Selection Rate'], line_dash='dash',
              annotation_text='Gesamtdurchschnitt')
fig.update_layout(showlegend=False, height=350)
fig.show()


Das Modell sagt 'Survived' für die 1. Klasse deutlich häufiger voraus als für die 3. Klasse.
Typisch: Selection Rate 1. Klasse ~65%, 3. Klasse ~20% — eine Spanne von ~45 Prozentpunkten.

**Fazit:** Das Modell ist *accurate* aber nicht *fair*. Es gibt keine Möglichkeit,
die historischen Klassenunterschiede zu entfernen ohne die Vorhersagequalität zu senken —
das Modell lernt die historische Realität, nicht eine normative Gleichbehandlung.


# B | Robustheit & Stabilität von Erklärungen 🔄
---

**Warum Robustheit prüfen?**

LIME basiert auf stochastischen Perturbationen — zwei Läufe mit unterschiedlichem `random_state`
können verschiedene Erklärungen liefern. Bei Compliance- oder Audit-Anforderungen ist
eine *stabile* Erklärung Pflicht.

In der Praxis relevant wenn: Erklärungen dokumentiert oder gegenüber Aufsichtsbehörden
nachgewiesen werden müssen.


In [ ]:
# LIME-Stabilität: 20 Läufe mit verschiedenen Seeds
feature_names = data_train.columns.tolist()
all_weights = {feat: [] for feat in feature_names}

# Wrapper: LIME uebergibt numpy arrays, RF erwartet DataFrame
def predict_proba_safe(X):
    return model.predict_proba(DataFrame(X, columns=feature_names))

for seed in range(20):
    exp_lime = LimeTabularExplainer(
        data_train.values,
        feature_names=feature_names,
        class_names=['Not Survived', 'Survived'],
        mode='classification',
        random_state=seed
    )
    exp = exp_lime.explain_instance(
        rose.iloc[0].values, predict_proba_safe, num_features=5
    )
    for feat_rule, weight in exp.as_list(label=1):
        for fn in feature_names:
            if fn in feat_rule:
                all_weights[fn].append(weight)
                break

print('LIME-Stabilitätsanalyse (20 Seeds, Rose):')
for feat, weights in all_weights.items():
    if weights:
        print(f'  {feat:8s}: MW={np.mean(weights):+.4f}  Std={np.std(weights):.4f}  n={len(weights)}')


In [ ]:
# Boxplot: LIME-Gewichte über 20 Seeds
rows = [{'Feature': f, 'Gewicht': w}
        for f, ws in all_weights.items() for w in ws]

df_stab = DataFrame(rows)
feat_order = (df_stab.groupby('Feature')['Gewicht']
              .apply(lambda x: abs(x).mean())
              .sort_values().index.tolist())

fig = px.box(
    df_stab, x='Gewicht', y='Feature', orientation='h',
    title='LIME-Stabilität: Gewichtsverteilung über 20 Seed-Läufe (Rose)',
    labels={'Gewicht': 'LIME-Gewicht', 'Feature': 'Feature'},
    category_orders={'Feature': feat_order}
)
fig.add_vline(x=0, line_dash='dash', line_color='gray')
fig.update_layout(height=350)
fig.show()


Die Print-Ausgabe zeigt die Stabilität pro Feature ueber 20 Seed-Läufe:

- **`sex`**: hoechster MW (ca. +0.32), kleinste Std (ca. 0.002) → stabilstes und wichtigstes Feature
- **`pclass`**: zweithöchster MW (ca. +0.17), ebenfalls stabil (ca. 0.003)
- **`age`**: mittlerer MW (ca. +0.08), etwas mehr Streuung → Einfluss stabil, aber moderater
- **`sibsp`/`parch`**: kleine MW, kleine Std → konsistent unwichtig

**Schmale Box** im Boxplot = konsistentes Gewicht ueber alle Seeds (Feature wird verlässlich erklärt)

Typischer Fehler: LIME-Erklärungen als 'die' Erklärung präsentieren ohne Hinweis auf Varianz.
Für diesen Datensatz ist LIME für Rose stabil — weil sex und pclass so stark dominieren,
dass der LIME-Algorithmus trotz Zufälligkeit fast immer dasselbe Ergebnis liefert.


In [ ]:
# SHAP-Stabilitaet: verschiedene Train-Test-Splits
shap_runs = []

for seed in range(10):
    X_tr, X_te, y_tr, _ = train_test_split(
        data, target, test_size=0.20, stratify=target, random_state=seed
    )
    X_tr = X_tr.copy(); X_te = X_te.copy()
    enc = OrdinalEncoder().fit(X_tr[cat_col])
    X_tr[cat_col] = enc.transform(X_tr[cat_col])
    X_te[cat_col] = enc.transform(X_te[cat_col])

    m = RandomForestClassifier(
        n_estimators=100, max_depth=5, min_samples_split=10,
        min_samples_leaf=5, random_state=seed
    ).fit(X_tr, y_tr)

    sv = shap.TreeExplainer(m).shap_values(X_te)
    # SHAP >= 0.42 gibt 3D-Array (n, features, classes), aeltere Versionen Liste
    if isinstance(sv, list):
        sv1 = sv[1]
    elif hasattr(sv, 'ndim') and sv.ndim == 3:
        sv1 = sv[:, :, 1]
    else:
        sv1 = sv
    shap_runs.append(np.abs(sv1).mean(axis=0))

df_shap_stab = DataFrame(shap_runs, columns=data_train.columns)
df_melt = df_shap_stab.melt(var_name='Feature', value_name='|SHAP|')
feat_ord = df_shap_stab.mean().sort_values().index.tolist()

fig = px.box(
    df_melt, x='|SHAP|', y='Feature', orientation='h',
    title='SHAP Global: Stabilitaet ueber 10 verschiedene Train-Test-Splits',
    labels={'|SHAP|': 'Mittlerer |SHAP-Wert|', 'Feature': 'Feature'},
    category_orders={'Feature': feat_ord}
)
fig.update_layout(height=350)
fig.show()


Die Boxen für SHAP (ueber 10 verschiedene Splits) sind breiter als die LIME-Boxen —
das liegt nicht am SHAP-Algorithmus selbst, sondern daran, dass **verschiedene Train-Test-Splits
leicht verschiedene Modelle ergeben**, die dann unterschiedliche SHAP-Werte produzieren.

LIME dagegen wird immer auf demselben Modell ausgefuehrt (nur der Perturbations-Seed variiert) —
daher ist die LIME-Varianz für diesen Datensatz kleiner.

Die **Rangfolge** (sex > pclass > age > sibsp > parch) bleibt ueber alle 10 Splits stabil —
das ist das entscheidende Zeichen für hohe Erklärungsqualität.


# C | Surrogate Models & EBM 🌲
---

**Zwei Alternativen zu 'Black Box nachträglich erklären':**

| Ansatz | Idee | Vorteil |
|--------|------|--------|
| **Global Surrogate** | Einfaches Modell auf die *Vorhersagen* des RF trainieren | Modellstruktur direkt lesbar |
| **EBM** | Direkt interpretierbares ML-Modell — kein Erklärungsschritt nötig | Kein Fidelity-Problem |

Grenze: Ein Surrogate-Modell erklärt das Black-Box-Modell nur näherungsweise.
Die Qualität hängt von der **Fidelity** ab — wie gut stimmt es mit dem RF überein?


In [ ]:
# Global Surrogate: Decision Tree auf RF-Vorhersagen trainieren
acc_rf   = accuracy_score(target_test, model.predict(data_test))
surrogate = DecisionTreeClassifier(max_depth=4, random_state=42)
surrogate.fit(data_train, model.predict(data_train))

fidelity = accuracy_score(model.predict(data_test), surrogate.predict(data_test))
acc_surr = accuracy_score(target_test, surrogate.predict(data_test))

print(f'RandomForest Accuracy:       {acc_rf*100:.2f}%')
print(f'Surrogate Accuracy:          {acc_surr*100:.2f}%')
print(f'Fidelity (Surrogate vs. RF): {fidelity*100:.2f}%')


Tatsächliche Ergebnisse:
- **RF Accuracy: 77.86%** | **Surrogate Accuracy: 76.34%** — Verlust nur 1.5%
- **Fidelity: 98.47%** — Das Surrogate trifft in 98.5% der Fälle dieselbe Entscheidung
  wie der RandomForest. Das ist sehr hoch.

Eine Fidelity von 98.5% bedeutet: Die Entscheidungsregeln des 4-stufigen Surrogate-Baums
erklären den 200-Baum-RandomForest fast vollständig — Zeichen dafür, dass die
Modelllogik tatsächlich durch einfache Regeln (sex + pclass) dominiert wird.


In [ ]:
# Surrogate Feature Importance vs. RF Feature Importance
df_comp = DataFrame({
    'Feature': data_train.columns,
    'RandomForest': model.feature_importances_,
    'Surrogate':    surrogate.feature_importances_
}).melt(id_vars='Feature', var_name='Modell', value_name='Importance')
df_comp = df_comp.sort_values('Importance')

fig = px.bar(
    df_comp, x='Importance', y='Feature', color='Modell',
    orientation='h', barmode='group',
    title='Feature Importance: RandomForest vs. Surrogate Decision Tree',
    labels={'Importance': 'Wichtigkeit', 'Feature': 'Feature'},
)
fig.update_layout(height=350)
fig.show()


Das Diagramm vergleicht die Feature Importance von RF und Surrogate direkt:

- Stimmen die Balken gut überein, erklärt das Surrogate den RF **verlässlich**
- Weichen sie stark ab, modelliert das Surrogate eine andere Logik als der RF

Typisch: `sex` und `pclass` dominieren in beiden Modellen — das Surrogate
reproduziert die RF-Logik für die wichtigsten Features korrekt.


In [ ]:
# EBM trainieren und mit RF vergleichen
acc_rf  = accuracy_score(target_test, model.predict(data_test))
ebm     = ExplainableBoostingClassifier(random_state=42)
ebm.fit(data_train, target_train)

acc_ebm = accuracy_score(target_test, ebm.predict(data_test))
print(f'EBM Accuracy: {acc_ebm*100:.2f}%  |  RF: {acc_rf*100:.2f}%')


In [ ]:
# EBM Feature Importance
ebm_global = ebm.explain_global()
ebm_data   = ebm_global.data()

df_ebm = DataFrame({
    'Feature':    ebm_data['names'],
    'Importance': ebm_data['scores']
}).sort_values('Importance')

fig = px.bar(
    df_ebm, x='Importance', y='Feature', orientation='h',
    title='EBM: Feature Importance (direkt interpretierbar)',
    labels={'Importance': 'Wichtigkeit', 'Feature': 'Feature'},
)
fig.update_layout(height=350)
fig.show()


Die EBM Feature Importance zeigt: `sex` und `pclass` dominieren — konsistent mit RF und SHAP.

Der Unterschied zum RF: Bei EBM stammt diese Importance direkt aus der Modellstruktur
(Gradient Boosting mit monotonen Terme), nicht aus einem nachträglichen SHAP-Aufruf.
Kein Fidelity-Problem, kein Erklärungsschritt nötig.


Typisch: **EBM ~76%**, **RF ~78%** — Unterschied nur ~2 Prozentpunkte.

EBM tauscht ~2% Accuracy gegen vollständige Interpretierbarkeit:
Die Feature-Importances im nächsten Diagramm stammen direkt aus dem Modell,
nicht aus einem nachträglichen Erklärungsschritt wie SHAP oder LIME.


# D | SHAP Interaction Values 🔗
---

**Was sind SHAP Interaction Values?**

Normale SHAP-Werte messen den individuellen Beitrag jedes Features.
SHAP Interaction Values zeigen zusätzlich, ob zwei Features **gemeinsam** stärker wirken
als ihre Einzelbeiträge — sogenannte Feature-Interaktionen.

| SHAP-Variante | Was wird gemessen? |
|---------------|--------------------|
| Normaler SHAP | Individueller Beitrag von Feature X |
| Interaction SHAP | Beitrag von Feature X wenn Feature Y bekannt ist |

In der Praxis relevant wenn: Man verstehen will, ob bestimmte Feature-Kombinationen
(z.B. sex=weiblich + pclass=1) synergistisch wirken.


In [ ]:
# SHAP Interaction Values (auf 100 Samples begrenzt)
if 'shap_explainer' not in vars() or shap_explainer is None:
    shap_explainer = shap.TreeExplainer(model)

print('Berechne SHAP Interaction Values...')
iv = shap_explainer.shap_interaction_values(data_test.iloc[:100])

# SHAP-Version-Check: Liste / 3D / 4D
if isinstance(iv, list):
    iv1 = iv[1]              # aeltere SHAP: Liste [class0, class1]
elif hasattr(iv, 'ndim') and iv.ndim == 4:
    iv1 = iv[:, :, :, 1]    # neuere SHAP: 4D (n, feat, feat, class)
elif hasattr(iv, 'ndim') and iv.ndim == 3:
    iv1 = iv                 # bereits klassenspezifisch
else:
    iv1 = iv

mean_iv = np.abs(iv1).mean(axis=0)  # (features x features)
df_iv   = DataFrame(mean_iv,
                    index=data_train.columns,
                    columns=data_train.columns)

print(f'iv Shape:   {np.array(iv).shape}')
print(f'iv1 Shape:  {iv1.shape}')
display(df_iv.round(4))


In [ ]:
# Heatmap der Interaktionswerte
fig = px.imshow(
    df_iv,
    title='SHAP Interaction Values: Feature-Interaktionen (Klasse: Survived)',
    labels=dict(color='|SHAP Interaction|'),
    color_continuous_scale='Blues',
    text_auto='.3f'
)
fig.update_layout(height=420)
fig.show()


Die Heatmap macht Interaktionsmuster visuell erkennbar:

**Dunkle Felder** (hohe Werte) markieren Feature-Paare, die das Modell gemeinsam
stärker gewichtet als ihre Einzelbeitrage. `sex × pclass` (Wert: 0.0223) ist der
dunkelste Off-Diagonal-Eintrag — die stärkste Feature-Interaktion.

Hinweis zur Farbskala: In der Blues-Palette gilt **dunkel = hoch, hell = niedrig**.
Die Diagonale ist am dunkelsten (individueller SHAP-Effekt), Off-Diagonale heller.
Das Balkendiagramm in der nächsten Zelle sortiert die Paare nach Interaktionsstärke.


Konkrete Werte aus der Tabelle:

| Paar | Interaction-Wert |
|------|------------------|
| sex × pclass | **0.0223** — stärkste Off-Diagonale |
| sex × parch | 0.0098 |
| age × sex | 0.0081 |
| age × pclass | 0.0058 |

- **Diagonale**: individueller SHAP-Effekt (sex=0.1528 ist der dominante Faktor)
- **Off-Diagonale sex×pclass=0.0223**: Diese Kombination wirkt stärker als ihre Einzelbeiträge
  — weiblich + 1. Klasse ist synergistisch günstiger als die Summe der Einzeleffekte


In [ ]:
# Top-Interaktionspaare als Balkendiagramm
feats = data_train.columns.tolist()
pairs = [
    {'Paar': feats[i] + ' x ' + feats[j], 'Interaktion': float(mean_iv[i, j])}
    for i in range(len(feats)) for j in range(i+1, len(feats))
]
df_pairs = DataFrame(pairs).sort_values('Interaktion')

fig = px.bar(
    df_pairs, x='Interaktion', y='Paar', orientation='h',
    title='Staerkste Feature-Interaktionen (SHAP)',
    labels={'Interaktion': '|SHAP Interaction|', 'Paar': 'Feature-Paar'},
)
fig.update_layout(height=350)
fig.show()


Das stärkste Interaktionspaar ist typischerweise **`sex × pclass`** — beide Features
werden vom Modell nicht unabhängig bewertet: Wer weiblich *und* 1. Klasse ist,
hat eine überproportional hohe Überlebenschance gegenüber Einzeleffekten.

Paare mit Interaktionswert nahe 0 (z.B. `sibsp × parch`) wirken unabhängig voneinander.


# E | Kausalität vs. Korrelation ⚠️
---

**Zentrale Einschränkung aller XAI-Methoden:**

SHAP, LIME, ALE und alle anderen Methoden erklären **Modelllogik** — keine Kausalzusammenhänge.

| Was XAI erklärt | Was XAI **nicht** erklärt |
|--|--|
| Warum hat das *Modell* so entschieden? | Warum überlebten Frauen häufiger? |
| Welche Features nutzt das Modell? | Welche Features *verursachen* das Ergebnis? |
| Wie würde sich die *Vorhersage* ändern? | Wie würde sich das *Ergebnis* ändern? |


In [ ]:
# Demo: SHAP-Wert fuer Rose
if 'shap_explainer' not in vars() or shap_explainer is None:
    shap_explainer = shap.TreeExplainer(model)

rose_shap = shap_explainer(rose)
sv_rose   = rose_shap.values[0, :, 1] if rose_shap.values.ndim == 3 else rose_shap.values[0]

print('SHAP-Werte fuer Rose:')
for feat, val in zip(data_train.columns, sv_rose):
    print(f'  {feat:8s}: {val:+.4f}')
print()
print('Was SHAP sagt:     sex=0 erhoeht die Vorhersage um ca. +0.21')
print('Was SHAP nicht sagt:')
print('  - Ob Geschlecht kausal fuer das Ueberleben war')
print('  - Ob Bootszuteilung als Confounder dazwischenliegt')
print('  - Ob ein anderes Modell dieselbe Korrelation nutzen wuerde')


**Das Mediator-Problem am Titanic-Beispiel:**

```
sex  ──→  Bootszuteilung  ──→  Überleben
          ↑
       pclass
```

SHAP sieht: `sex` korreliert mit Überleben → hoher SHAP-Wert.
SHAP sieht nicht: Die kausale Kette läuft ueber Bootszuteilung als **Mediator**
(nicht Confounder — ein Mediator liegt *auf dem Wirkungspfad*, ein Confounder liegt *daneben*).

**Typischer Fehler:** SHAP-Werte als Kausalaussagen interpretieren:
❌ 'Weil jemand weiblich ist, überlebt er.' — Nein, das Modell nutzt die Korrelation.

**Faustregel:** XAI + kausales Denken = gute XAI-Praxis. XAI allein = Modellbeschreibung.


In [ ]:
# Proxy-Demo: Was passiert wenn 'sex' entfernt wird?
data_no_sex      = data_train.drop(columns=['sex'])
data_test_no_sex = data_test.drop(columns=['sex'])

model_ns = RandomForestClassifier(
    n_estimators=100, max_depth=5, min_samples_split=10,
    min_samples_leaf=5, random_state=42
).fit(data_no_sex, target_train)

acc_ns  = accuracy_score(target_test, model_ns.predict(data_test_no_sex))
acc_rf2 = accuracy_score(target_test, model.predict(data_test))

sv_ns = shap.TreeExplainer(model_ns).shap_values(data_no_sex)
if isinstance(sv_ns, list):
    sv_ns1 = sv_ns[1]
elif hasattr(sv_ns, 'ndim') and sv_ns.ndim == 3:
    sv_ns1 = sv_ns[:, :, 1]
else:
    sv_ns1 = sv_ns

imp_ns = DataFrame({
    'Feature': data_no_sex.columns,
    '|SHAP|':  np.abs(sv_ns1).mean(axis=0)
}).sort_values('|SHAP|')

print(f'Accuracy ohne sex: {acc_ns*100:.2f}%  (mit sex: {acc_rf2*100:.2f}%)')
print()
print('Feature Importance ohne sex:')
display(imp_ns)


Wenn `sex` entfernt wird, sinkt die Accuracy kaum — `pclass` übernimmt als Proxy.
Das Modell findet die nächstbeste korrelierte Variable.

Das zeigt: Entfernen eines korrelierten Features löst das kausale Problem nicht.
Es verschiebt es nur auf einen anderen Proxy.


# F | Anchors ⚓
---

**Was sind Anchors?**

Anchors sind lokale **Wenn-dann-Regeln**, die zeigen unter welchen Bedingungen
eine Vorhersage *stabil bleibt* — unabhängig davon wie andere Features variieren:

> IF sex = weiblich AND pclass = 1. Klasse THEN Survived (Präzision: 97%)

| Begriff | Bedeutung |
|---------|----------|
| **Präzision** | Anteil korrekter Vorhersagen unter dieser Regel (≥ threshold) |
| **Coverage** | Anteil der Datenpunkte, auf die diese Regel zutrifft |

**Unterschied zu LIME/SHAP:** LIME/SHAP zeigen *wie stark* ein Feature wirkt;
Anchors zeigen *welche Bedingungen* die Vorhersage garantieren.

Grenze: Anchors können sehr eng (hohe Präzision, niedrige Coverage) oder
sehr breit (niedrige Präzision, hohe Coverage) sein.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

try:
    from alibi.explainers import AnchorTabular

    anchor_exp = AnchorTabular(
        predictor=model.predict,
        feature_names=data_train.columns.tolist(),
    )
    anchor_exp.fit(data_train.values, disc_perc=(25, 50, 75))

    for person, name in [(rose, 'Rose'), (jack, 'Jack')]:
        result = anchor_exp.explain(person.values[0], threshold=0.90)
        print(f'Anchor fuer {name} (Praezision >= 90%):')
        print('  Regeln:     ' + (' AND '.join(result.anchor) or '(keine Regel noetig)'))
        print(f'  Praezision: {result.precision:.3f}')
        print(f'  Coverage:   {result.coverage:.3f}')
        print()

except ImportError:
    print('alibi nicht installiert.')
    print('Installation: !pip install alibi')
    print()
    print('Manuelle Anchor-Approximation (regelbasiert):')
    rules = [
        {'Bedingung': 'sex=0 (weiblich)',                'Praezision': None, 'Vorhersagen': None},
        {'Bedingung': 'sex=0 AND pclass=0 (1. Kl.)',    'Praezision': None, 'Vorhersagen': None},
    ]
    for r in rules:
        mask = data_test['sex'] == 0
        if 'pclass' in r['Bedingung']:
            mask = mask & (data_test['pclass'] == 0)
        preds = model.predict(data_test[mask])
        prec  = preds.mean()
        cov   = mask.mean()
        print(f"  '{r['Bedingung']}'")
        print(f'  Praezision: {prec:.3f}  Coverage: {cov:.3f}')
        print()


Tatsächliche Anchor-Regeln aus dem Output:

**Rose:** `sex <= 0.00 AND pclass <= 2.00`
- Übersetzt: weiblich **und** (1. oder 2. Klasse)
- **Präzision: 0.924**: In 92.4 % der perturbierter Instanzen unter dieser Regel
  trifft das **Modell** dieselbe Vorhersage ('Survived') — nicht Ground-Truth-Korrektheit
- **Coverage: 0.346**: 34.6 % der zufällig perturbierten Instanzen erfüllen diese Regel

**Jack:** `pclass > 2.00` (der Zusatz `sex <= 1.00` ist tautologisch, da sex ∈ {0,1})
- **Präzision: 0.980**: Das Modell bleibt in 98 % der Perturbationen bei 'nicht überlebt'
- **Coverage: 1.000**: Alle perturbierten Instanzen erfüllen `pclass > 2.00`

Wichtig: Anchor-Präzision misst **Modellstabilität**, nicht Treffsicherheit gegenueber
der Realitaet. Coverage ist Perturbations-Coverage, nicht Testset-Abdeckung.


# G | Example-Based Explanations 📋
---

**Was sind Example-Based Explanations?**

Statt mathematischer Formeln werden echte Datenpunkte als Erklärung genutzt:

| Methode | Idee |
|---------|------|
| **k-NN Exemplare** | 'Diese Vorhersage ähnelt diesen historischen Fällen' |
| **Prototypen** | Repräsentative Fälle pro Klasse aus dem Trainingsset |
| **Gegenbeispiele** | Ähnliche Fälle mit *anderer* Vorhersage (z.B. Counterfactuals) |

**Vorteil:** Menschen verstehen Beispiele oft intuitiver als Formeln.

In der Praxis relevant wenn: Entscheidungen Betroffenen erklärt werden müssen,
z.B. Kreditentscheidungen: 'Ähnliche Personen erhielten keinen Kredit weil...'


<p><font color='black' size="5">
Skalierung für Ähnlichkeitsanalysen
</font></p>

Random Forest und Decision Tree benötigen keine Skalierung. Die folgende Skalierung wird nur für k-NN-ähnliche Distanzvergleiche in den Example-Based Explanations verwendet.

In [ ]:
# k-NN: Ähnlichste Trainingsfälle finden (auf skalierten Daten)
# Hinweis: scaler wird in der Prototypen-Zelle definiert (vorher ausführen)
from sklearn.preprocessing import StandardScaler as _SC
_scaler_knn = _SC().fit(data_train)
X_train_sc  = _scaler_knn.transform(data_train)

nn = NearestNeighbors(n_neighbors=5, metric='euclidean')
nn.fit(X_train_sc)

def zeige_aehnliche(person_df, name):
    person_sc = _scaler_knn.transform(person_df)
    dists, idxs = nn.kneighbors(person_sc)
    similar = data_train.iloc[idxs[0]].copy()
    similar['survived'] = target_train.iloc[idxs[0]].values
    similar['Distanz']  = dists[0].round(3)
    similar['sex']    = similar['sex'].map({0.0: 'weiblich', 1.0: 'männlich'})
    similar['pclass'] = similar['pclass'].map({1.0: '1.Kl', 2.0: '2.Kl', 3.0: '3.Kl'})
    similar['survived'] = similar['survived'].map({0: 'nicht', 1: 'ja'})
    print(f'Ähnlichste Fälle für {name} (skaliert):')
    display(similar[['age', 'sex', 'pclass', 'sibsp', 'parch', 'survived', 'Distanz']])
    print()

zeige_aehnliche(rose, 'Rose')
zeige_aehnliche(jack, 'Jack')


**Rose** (22, weiblich, 1. Klasse): Die 5 nächsten Nachbarn sind überwiegend
ebenfalls weiblich in der 1. Klasse — Rose liegt klar im Zentrum ihrer Gruppe.
Das Überleben der Nachbarn ist nicht einheitlich, da Alter und Familiensituation
kleine Unterschiede erzeugen. Die Vorhersage 62% spiegelt diese Unsicherheit wider.

**Jack** (23, männlich, 3. Klasse): Alle 5 Nachbarn haben nicht überlebt —
Jack liegt klar im Zentrum der 'nicht überlebt'-Klasse. Die 7% Überlebenschance
ist durch die Nachbarschaft gut begruendet.

Gemischte Nachbarschaft = Datenpunkt liegt nahe der Entscheidungsgrenze.
Einheitliche Nachbarschaft = das Modell ist in diesem Bereich sicher.


In [ ]:
# Prototypen: repraesentativster Fall pro Klasse
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler().fit(data_train)
X_sc   = scaler.transform(data_train)

for label, label_name in [(1, 'Ueberlebt'), (0, 'Nicht ueberlebt')]:
    mask      = target_train.values == label
    centroid  = X_sc[mask].mean(axis=0)
    dists     = np.linalg.norm(X_sc[mask] - centroid, axis=1)
    proto_idx = np.argmin(dists)
    proto     = data_train[mask].iloc[proto_idx].copy()
    proto['sex']    = 'weiblich' if proto['sex'] == 0 else 'maennlich'
    # pclass ist NICHT OrdinalEncoded -> Werte 1/2/3
    proto['pclass'] = {1.0: '1. Kl.', 2.0: '2. Kl.', 3.0: '3. Kl.'}.get(proto['pclass'], '?')
    print(f'Prototyp fuer [{label_name}]:')
    for k, v in proto.items():
        print(f'  {k:8s}: {v}')
    print()


Tatsächliche Prototypen (Klassen-Zentroid-nächster Punkt):

- **Prototyp 'Überlebt'**: weiblich, ca. 28 Jahre, 2. Klasse, allein reisend
- **Prototyp 'Nicht überlebt'**: männlich, ca. 30 Jahre, 2. Klasse, allein reisend

Beide Prototypen liegen in der 2. Klasse — die Klasse unterscheidet sie hier nicht.
Der **einzige klare Unterschied ist das Geschlecht**.

Das liegt daran, dass das Klassen-Zentroid im standardisierten Merkmalsraum
nicht unbedingt der häufigsten Klasse entspricht, sondern dem geometrischen
Mittelpunkt aller Trainingsdaten der jeweiligen Klasse.
Prototypen sind daher kein 'typischer Fall' im alltagssprachlichen Sinne,
sondern der dem Zentroid am nächsten liegende reale Datenpunkt.


In [ ]:
# Gegenbeispiele für Jack: ähnliche Fälle die überlebt haben (skaliert)
jack_sc = _scaler_knn.transform(jack)

survived_mask = target_train.values == 1
X_surv_sc     = X_train_sc[survived_mask]
X_surv_raw    = data_train[survived_mask].values

dists_surv = np.linalg.norm(X_surv_sc - jack_sc, axis=1)
top5_idx   = np.argsort(dists_surv)[:5]

df_gegen = DataFrame(X_surv_raw[top5_idx], columns=data_train.columns)
df_gegen['sex']    = df_gegen['sex'].map({0.0: 'weiblich', 1.0: 'männlich'})
df_gegen['pclass'] = df_gegen['pclass'].map({1.0: '1.Kl', 2.0: '2.Kl', 3.0: '3.Kl'})
df_gegen['Distanz'] = dists_surv[top5_idx].round(3)

print('Jack: Ähnlichste Fälle die überlebt haben (Gegenbeispiele, skaliert):')
display(df_gegen[['age', 'sex', 'pclass', 'sibsp', 'parch', 'Distanz']])


Die Tabelle zeigt 5 Trainingsfälle, die Jack ähneln
und trotzdem überlebt haben.

Überraschend: Alle 5 Gegenbeispiele sind ebenfalls **männlich und 3. Klasse** —
sie unterscheiden sich von Jack fast ausschließlich im **Alter** (22–27 statt 23 Jahre).

Was das bedeutet: Es gab real ähnliche Männer in 3. Klasse, die überlebt haben.
Das Modell gewichtet diese Grenzfälle mit ~7 % Überlebenschance — korrekt abgewägt,
aber kein einzelnes Feature springt als 'der Unterschied' heraus.

Grenze: k-NN-Gegenbeispiele zeigen reale historische Fälle — keine garantiert minimale
Änderung. Der Counterfactual-Abschnitt in b900 sucht explizit nach der kleinsten Änderung.


# H | Zusammenfassung 🔬
---

| Abschnitt | Methode | Scope | Wichtigste Erkenntnis | Einsteiger |
|-----------|---------|-------|-----------------------|------------|
| **A** | Fairness & Bias | Global | Accuracy ≠ Fairness; Gruppen können benachteiligt sein | ⭐⭐⭐⭐ |
| **B** | Robustheit | Global/Lokal | LIME schwankt, SHAP ist stabiler | ⭐⭐⭐ |
| **C** | Surrogate & EBM | Global | EBM: direkt interpretierbar ohne Post-hoc | ⭐⭐⭐ |
| **D** | SHAP Interactions | Global | Feature-Kombinationen können synergistisch wirken | ⭐⭐⭐⭐ |
| **E** | Kausalität | Konzept | XAI erklärt Modelllogik, keine Kausalität | ⭐⭐⭐⭐⭐ |
| **F** | Anchors | Lokal | Wenn-dann-Regeln für stabile Vorhersagen | ⭐⭐⭐ |
| **G** | Example-Based | Lokal | Erklärung über reale Beispiele und Prototypen | ⭐⭐⭐⭐ |


**Weiterführende Ressourcen:**

- **Fairlearn:** https://fairlearn.org/
- **alibi (Anchors, Counterfactuals):** https://docs.seldon.io/projects/alibi/
- **InterpretML / EBM:** https://interpret.ml/
- **Causal XAI:** Pearl, J. 'The Book of Why' (2018)
- **Anchors Paper:** Ribeiro et al. 'Anchors: High-Precision Model-Agnostic Explanations' (2018)
- **SHAP Interaction Values:** Lundberg et al. 'Consistent Individualized Feature Attribution' (2018)
